#개요
Olist 데이터셋 가지고 text-to-sql FineTuning용 데이터셋 만들기.

### 전체 flow

1. 환경 설정
- df 이름, table 이름

2. LLM 실행
- DDL Statement
- 칼럼 설명
- 칼럼별 unique한 값 예시
- base dataset 질문 예시 (실제 질문처럼 하기 위해)

3. 응답 파싱

4. 질문 말투 다양화
- 칼럼명 간접 언급, 명사구 질문, 종결 어미 변경

5. 결과 저장
- DDL문 뒤 INSERT INTO - VALUES - 랜덤 개수 추가



#환경설정

In [1]:
!pip install langchain-openai langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3


In [2]:
import json
import re
import pandas as pd
import random
import sqlite3
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [3]:
os.environ["OPENAI_API_KEY"] = ""

In [4]:
llm = ChatOpenAI(model = "gpt-5", temperature = 0.3)
llm_sql = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)
llm_small = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0.7)

목표

instruction : "DDL Statements:DDL문\n입력:한글프롬프트"

input : ""

output : SQL 문

In [5]:
with open('/content/gretelai_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

[
  {
    "instruction": "DDL statements:\nCREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');\n입력 텍스트: 각 판매원이 판매한 목재의 총량은 얼마이며, 판매원에 따라 정렬되어 있나요?\n\n위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.",
    "input": "",
    "output": "쿼리 작성: SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;"
  },
  {
    "instruction": "DDL statements:\nCREATE TABLE equipment_maintenance (equipment_type VARCHAR(255), maintenance_frequency INT);\n입력 텍스트: 장비 유형과 해당 장비의 전체 유지 보

데이터셋 로드

In [6]:
#Olist 데이터
df_customers        = pd.read_csv('olist_customers_dataset.csv', dtype={'customer_zip_code_prefix': str})
df_geolocation      = pd.read_csv('olist_geolocation_dataset.csv', dtype={'geolocation_zip_code_prefix': str})
df_order_items      = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments   = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews    = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders           = pd.read_csv('olist_orders_dataset.csv')
df_products         = pd.read_csv('olist_products_dataset.csv')
df_sellers          = pd.read_csv('olist_sellers_dataset.csv', dtype={'seller_zip_code_prefix': str})

In [7]:
#base text-to-sql 데이터
!wget https://raw.githubusercontent.com/leejunho12316/LLaMA-Factory/main/data/text_to_sql_data.json

--2026-06-22 11:52:30--  https://raw.githubusercontent.com/leejunho12316/LLaMA-Factory/main/data/text_to_sql_data.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3676178 (3.5M) [application/octet-stream]
Saving to: ‘text_to_sql_data.json’

text_to_sql_data.js 100%[===================>]   3.51M  --.-KB/s    in 0.07s   

2026-06-22 11:52:31 (50.2 MB/s) - ‘text_to_sql_data.json’ saved [3676178/3676178]



In [8]:
with open('text_to_sql_data.json') as f:
  base_data = json.load(f)

In [9]:
print(base_data[25]['instruction'])

입력 텍스트: 아프리카에서 활동하는 모든 식량 정의 단체와 그들이 진행한 프로젝트 수를 나열하세요.

DDL statements:
CREATE TABLE food_justice_orgs (org_id INT, org_name TEXT, country TEXT, num_projects INT); INSERT INTO food_justice_orgs (org_id, org_name, country, num_projects) VALUES (1, 'Org A', 'Kenya', 10), (2, 'Org B', 'Nigeria', 7), (3, 'Org C', 'South Africa', 15);

위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.


# 함수 모음

In [10]:
def parse_llm_output(llm_output: str, num_pairs: int) -> list[dict]:
    """
    LLM 출력을 파싱해 질문-SQL 딕셔너리를 담은 리스트로 반환

    Args:
        llm_output : LLM 출력 문자열
        num_pairs  : 파싱할 질문-SQL 쌍의 개수

    Returns:
        [{'Question': '...', 'SQL': '...'}, ...]
    """
    questions = re.findall(r'\[질문\]\s*(.*?)\s*(?=\[SQL\])', llm_output, re.DOTALL)
    sqls      = re.findall(r'\[SQL\]\s*(.*?)\s*(?=\[질문\]|$)',  llm_output, re.DOTALL)

    # 백틱 제거
    sqls = [re.sub(r'```sql|```', '', sql).strip() for sql in sqls]
    questions = [q.strip() for q in questions]

    result = []
    for i in range(min(num_pairs, len(questions), len(sqls))):
        result.append({
            'Question': questions[i],
            'SQL':      sqls[i]
        })

    return result

In [11]:
def _convert_to_sqlite(sql: str) -> str:
    """
    DB 종류에 따라 다른 SQL문을 LLM을 사용해 SQLite 문법으로 변환.
    execute_sql_on_db 에서만 사용하는 함수

    Args:
        sql : 변환할 SQL문

    Returns:
        SQLite 문법으로 변환된 SQL문
    """
    response = llm_sql.invoke(
        f"""다음 SQL문을 SQLite 문법으로 변환해줘.
반드시 SQL문만 출력하고 다른 설명은 절대 추가하지 마.
백틱이나 코드블록 없이 순수 SQL문만 출력해.

{sql}"""
    )
    return response.content.strip()


def execute_sql_on_db(parsed_results: list[dict], conn) -> list[dict]:
    """
    질문-SQL 딕셔너리를 담은 List를 받아 DB에 전체 실행해보기.
    """

    results = []

    for item in parsed_results:
        question  = item['Question']
        sql       = item['SQL'].rstrip(';')
        sql_sqlite = None

        # SQL 유형 판별
        sql_type = sql.strip().split()[0].upper()  # SELECT, UPDATE, DELETE, INSERT

        def run_sql(query):
            if sql_type == 'SELECT':
                # SELECT는 pd.read_sql_query() 사용
                return pd.read_sql_query(query, conn), 'success'
            else:
                # INSERT, UPDATE, DELETE는 cursor로 실행 후 롤백
                cursor = conn.cursor()
                cursor.execute(query)
                affected = cursor.rowcount
                conn.rollback()  # 실제 반영 안 되게 롤백
                return pd.DataFrame({'affected_rows': [affected]}), 'success'

        # 1차 시도: 원본 SQL 실행
        try:
            df_result, status = run_sql(sql)
        except Exception as e:
            df_result = None
            status    = f'error: {e}'

            # 2차 시도: LLM으로 SQLite 변환 후 재실행
            print(f"❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도")
            try:
                sql_sqlite        = _convert_to_sqlite(sql).rstrip(';')
                df_result, status = run_sql(sql_sqlite)
            except Exception as e2:
                df_result = None
                status    = f'❌error (변환 후에도 실패): {e2}'

        results.append({
            'Question'  : question,
            'SQL'       : sql,
            'SQL_SQLite': sql_sqlite,
            'Result'    : df_result,
            'Status'    : status
        })

        print(f"1. Question:   {question}")
        print(f"2. SQL 원본:\n{sql}")
        if sql_sqlite:
            print(f"2-1. SQL 변환:\n{sql_sqlite}")
        print(f"3. Status:     {status}")
        print("\n4. 실행 결과:\n")
        print(df_result if df_result is not None else "")
        print("\n")
        print("-" * 50)

    return results

In [12]:
#Prompt 예시 추가용 함수
def get_sample_values(df, n=3) -> str:
  """
  dataframe의 각 칼럼 별 unique한 값 중 랜덤으로 n개를 뽑은 결과 반환. Prompt 추가용.

  인수 :
    dataframe : olist 데이터프레임
    n : 랜덤으로 추출할 값 개수
  return :
    dataframe 각 칼럼에서 unique 한 값 중 랜덤으로 n개를 뽑은 결과 str
  """
  sample_text = ""
  for col in df.columns:
      uniques = df[col].dropna().unique().tolist()
      samples = random.sample(uniques, min(n, len(uniques)))  # 랜덤으로 n개 추출
      sample_text += f"{col} : {samples}\n"
  return sample_text

def get_sample_queries(base_data, n: int):
  """
  json 데이터 중 랜덤으로 n개의 query를 뽑은 결고 반환. Prompt 추가용
  """
  queries = [
    i.get('instruction').split('DDL statements:')[0].split('입력 텍스트:')[1].strip() for i in random.sample(base_data, n)
  ]
  return "\n".join(queries)

In [13]:
def df_to_insert_sql(df: pd.DataFrame, table_name: str) -> str:
    """
    DataFrame을 받아 해당 테이블에 대한 'INSERT INTO ~' SQL 구문만 생성하여 반환한다.

    - 모든 컬럼을 대상으로 VALUES를 작성한다.
    - 삽입되는 행(VALUES) 개수는 1~5개 중 랜덤하게 결정된다.
    - CREATE TABLE 구문은 포함하지 않고 INSERT INTO 부분만 리턴한다.

    Parameters
    ----------
    df : pd.DataFrame - DDL을 만들 대상 데이터
    table_name : str - 테이블 이름

    Returns
    -------
    str
        "INSERT INTO table_name (col1, col2, ...) VALUES (...), (...);" 형태의 문자열
    """
    if df.empty:
        raise ValueError("df가 비어 있어 INSERT 구문을 만들 수 없습니다.")

    columns = list(df.columns)

    # 1~5개 사이에서 랜덤하게 행 개수 결정 (df 행 수보다 클 수 없음)
    num_rows = random.randint(0, min(5, len(df)))
    if num_rows == 0:
      return ''
    sample_df = df.sample(n=num_rows).reset_index(drop=True)

    # 컬럼별로 숫자형인지 여부를 미리 판단 (값 포맷팅에 사용)
    is_numeric_col = {col: pd.api.types.is_numeric_dtype(df[col]) for col in columns}

    def format_value(col, val):
        if pd.isna(val):
            return "NULL"
        if is_numeric_col[col] and not isinstance(val, bool):
            # 정수/실수 그대로 출력
            return str(val)
        if isinstance(val, bool):
            return str(val).upper()  # TRUE / FALSE
        # 문자열, 날짜 등은 작은따옴표로 감싸고 내부 작은따옴표는 이스케이프
        escaped = str(val).replace("'", "''")
        return f"'{escaped}'"

    value_rows = []
    for _, row in sample_df.iterrows():
        values = ", ".join(format_value(col, row[col]) for col in columns)
        value_rows.append(f"({values})")

    columns_str = ", ".join(columns)
    values_str = ", ".join(value_rows)

    return f"INSERT INTO {table_name} ({columns_str}) VALUES {values_str};"



In [14]:
# #최종 저장용 함수
# def convert_to_gretel_format(total_result: list[dict], ddl: str) -> list[dict]:
#     """
#     total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

#     Args:
#         total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
#         ddl           : instruction에 넣을 DDL 문자열

#     Returns:
#         [{'instruction': ..., 'input': '', 'output': ...}, ...]
#     """
#     converted = []
#     for item in total_result:
#         question = item['Question']
#         sql      = item['SQL']

#         instruction = (
#             f"입력 텍스트: {question}\n\n"
#             f"DDL statements:\n{ddl}\n\n"
#             f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
#         )

#         converted.append({
#             "instruction": instruction,
#             "input"      : "",
#             "output"     : f"쿼리 작성: {sql}"
#         })

#     return converted

In [15]:
#최종 저장용 함수
def convert_to_gretel_format(total_result: list[dict], ddl: str, df: pd.DataFrame, table_name: str) -> list[dict]:
    """
    total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

    Args:
        total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
        ddl           : instruction에 넣을 DDL 문자열
        df            : INSERT INTO 문을 생성할 DataFrame
        table_name    : INSERT INTO 문에 사용할 테이블 명

    Returns:
        [{'instruction': ..., 'input': '', 'output': ...}, ...]
    """
    converted = []
    for item in total_result:
        question = item['Question']
        sql      = item['SQL']

        insert_sql = df_to_insert_sql(df, table_name)

        instruction = (
            f"입력 텍스트: {question}\n\n"
            f"DDL statements:\n{ddl}\n{insert_sql}\n\n"
            f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
        )

        converted.append({
            "instruction": instruction,
            "input"      : "",
            "output"     : f"쿼리 작성: {sql}"
        })

    return converted

In [44]:
import random
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ENDING_STYLES = ["~요?", "~까?", "~임?", "~나?", "~습니까?", "~나요?", "~가요?", "~니?", "~냐?"]

REPHRASE_SYSTEM_PROMPT = """당신은 text-to-SQL 데이터셋의 질문(Question) 문장 스타일을 다양화하는 어시스턴트입니다.
주어진 질문을 아래 기준에 맞게 자연스러운 한국어로 다시 작성하세요.

[기준 1] 컬럼명 직접/간접 언급
질문에 영문 컬럼명이나 변수명이 그대로 노출되어 있다면, 의미가 통하는 자연어 표현으로 바꾸세요.
예) "각 country_of_origin별 모든 satellites의 최대 거리는 얼마인가요?"
    -> "각 국가별로 지구 표면으로부터 모든 위성의 최대 거리는 얼마인가요?"
예) "country가 Africa인 모든 org_name 값과 그들이 진행한 num_projects 수를 나열하세요"
    -> "아프리카에서 활동하는 모든 식량 정의 단체와 그들이 진행한 프로젝트 수를 나열하세요."

[기준 2] 명사구 형태 변형 (지시가 있을 때만)
완전한 문장(서술어로 끝남) 대신, 명사구로 끝나는 형태로 바꾸세요.
예) "마을 변호사는 몇 명이었는가?" -> "마을변호사 인원 수"
예) "각 고객별 첫 구매 일시를 알고 싶습니다. 고객 ID와 첫 구매 타임스탬프를 반환해 주세요."
    -> "각 고객별 첫 구매 일시에 대한 고객 ID와 첫 구매 타임스탬프."
예) "2018년 2분기(Q2)에 구매된 주문들의 구매 시각부터 배송사 인계까지 평균 며칠이 걸렸는지 알려주세요"
    -> "2018년 2분기 구매 시각부터 배송사 인계까지 평균일."

[기준 3] 문장 종결 어미 변경 (문장형일 때만)
지정된 종결 어미를 사용해 문장을 자연스럽게 끝맺으세요. (~요?, ~까?, ~임?, ~나? 등)
단, 현재 문장에 종결어미를 적용했을 때 억지스럽거나 부자연스럽다면 다른 종결어미를 적용하세요.

예) 결제 수단별로 결제 승인까지 평균 몇 시간이 걸리는가요?
    -> 결제 수단별로 결제 승인까지 평균 몇 시간이 걸리나?
예) 2018년 6월에 예상 배송일을 넘겨 배송된 주문 중 결제 총액이 500 이상인 주문 ID와 며칠 늦었는지 알려주세요
    -> 2018년 6월에 예상 배송일을 넘겨 배송된 주문 중 결제 총액이 500 이상인 주문 ID와 며칠 늦었는지 알려줄 수 있나요?

[중요]
- 질문의 의미(조건, 대상 컬럼, 집계/필터 방식 등)는 절대 바꾸지 마세요. SQL과 매칭이 깨지면 안 됩니다.
- 결과는 변형된 질문 문장 하나만 출력하세요. 따옴표, 설명, 번호, "변형된 질문:" 같은 접두사를 붙이지 마세요."""

USER_PROMPT = """원본 질문: {question}

다음 지시를 반영해 질문을 다시 작성하세요.
- 명사구 형태로 변형: {use_noun_phrase}
- 문장으로 끝낼 경우 종결 어미: {ending_style}"""


def rephrase_questions(total_result: list[dict], llm, noun_phrase_ratio: float = 0.4) -> list[dict]:
    """
    total_result의 Question만 LLM을 통해 다양한 말투/표현으로 바꿔서 반환한다. (SQL은 그대로 유지)

    Args:
        total_result      : [{'Question': ..., 'SQL': ...}, ...]
        llm                : LangChain ChatModel (예: llm_small)
        noun_phrase_ratio  : 명사구 형태로 변형할 확률 (기본 40%)

    Returns:
        [{'Question': 변형된 질문, 'SQL': 원본 SQL}, ...]
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", REPHRASE_SYSTEM_PROMPT),
        ("human", USER_PROMPT),
    ])
    chain = prompt | llm | StrOutputParser()

    # row마다 명사구 여부 / 종결어미를 랜덤하게 지정해서 batch 입력 구성
    batch_inputs = []
    for item in total_result:
        use_noun_phrase = random.random() < noun_phrase_ratio
        ending_style = random.choice(ENDING_STYLES)
        batch_inputs.append({
            "question": item["Question"],
            "use_noun_phrase": "예 (명사구로 끝내기)" if use_noun_phrase else "아니오 (완전한 문장 유지)",
            "ending_style": "해당 없음 (명사구이므로 종결어미 사용 안 함)" if use_noun_phrase else ending_style,
        })

    # LangChain batch 호출로 한 번에 처리 (내부적으로 병렬 실행)
    try:
        rephrased_questions = chain.batch(batch_inputs, config={"max_concurrency": 5})
    except Exception as e:
        print(f"⚠️ batch 호출 중 오류 발생, 개별 호출로 재시도: {e}")
        rephrased_questions = []
        for inp in batch_inputs:
            try:
                rephrased_questions.append(chain.invoke(inp))
            except Exception:
                rephrased_questions.append(None)

    converted = []
    for item, new_question in zip(total_result, rephrased_questions):
        # 실패한 경우 원본 질문 유지
        question = new_question.strip() if new_question else item["Question"]
        converted.append({
            "Question": question,
            "SQL": item["SQL"],
        })
    return converted

# LLM 관련 설정

In [17]:
SYSTEM_PROMPT = """
#역할
당신은 Text-to-SQL을 수행해야합니다.
DDL 선언문, 칼럼 설명, 칼럼 값 예시, 질문 예시를 참고해 사용자가 할 법한 질문-SQL 쌍을 작성해주세요.
실제 사용자가 Text-to-SQL LLM에게 자연스럽게 물어볼 법한 질문과 그에 정확히 대응하는 SQL을 작성하세요.
질문-SQL쌍은 10개 생성하세요.

#최우선 중요 원칙
사람이 실제로 어떻게 질문할지 생각하세요. 그리고 그 질문에 정확히 대응하는 SQL 문을 작성하세요.
질문 예시를 적극적으로 참고하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.
4. SQL 작성시 주의
- BETWEEN : 기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다. 기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'
- 현재 날짜/시간 : 현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [18]:
HUMAN_PROMPT = """
# DDL 선언문
{DDL_Statement}

# 칼럼 설명
{column_descriptions}

# 칼럼 값 예시
{column_values}

# 질문 예시
{question_examples}

# history
{history}
"""

In [19]:
def generate_sql(DDL_Statement: str, column_descriptions: str, column_values: str, question_examples: str,
                 history : list[dict], is_multi=False) -> str:
    """
    DDL문과 컬럼 설명을 입력받아 LLM으로 질문-SQL 쌍을 생성

    Args:
        DDL_Statement       : 테이블 DDL 문
        column_descriptions : 컬럼 설명
        column_values       : 컬럼 값 예시 (함수 사용해서 들어감)
        question_examples   : 질문 예시들

    Returns:
        LLM이 생성한 질문-SQL 쌍 문자열
    """

    system_prompt = SYSTEM_PROMPT_MULTI if is_multi else SYSTEM_PROMPT

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
    ])

    chain = prompt | llm

    # history 중 질문만 str로 취합
    history_str = ""
    for res in total_result:
      history_str += res.get('Question') + "\n"

    response = chain.invoke({
        "DDL_Statement"      : DDL_Statement,
        "column_descriptions": column_descriptions,
        "column_values" : column_values,
        "question_examples" : question_examples,
        "history" : history_str
    })

    print(response.usage_metadata) # 캐싱 작동 확인용
    return response.content

#단일 SQL 문 생성


각 DDL문은 다음 프롬프트를 사용해 얻기.
(Human Prompt에 DF으로부터 값 예시 랜덤으로 넣는 로직 삭제하고 DDL 문의 INSERT INTO ~ VALUES로 대체하기)

```
DB에 대한 DDL문을 다음 예시처럼 한 줄로 작성해줘.
DDL문을 제외한 그 어떤 출력도 하지마.
DDL문만 출력해줘.

#테이블명
orders

#DB
order_id	customer_id	order_status	order_purchase_timestamp	order_approved_at	order_delivered_carrier_date	order_delivered_customer_date	order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7	9ef432eb6251297304e76186b10a928d	delivered	2017-10-02 10:56:33	2017-10-02 11:07:15	2017-10-04 19:55:00	2017-10-10 21:25:13	2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451	b0830fb4747a6c6d20dea0b8c802d7ef	delivered	2018-07-24 20:41:37	2018-07-26 03:24:27	2018-07-26 14:31:00	2018-08-07 15:27:45	2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d	41ce2a54c0b03bf3443c3d931a367089	delivered	2018-08-08 08:38:49	2018-08-08 08:55:23	2018-08-08 13:50:00	2018-08-17 18:06:29	2018-09-04 00:00:00

#예시
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
CREATE TABLE farmers_india (id INT, name VARCHAR(255), district_id INT, age INT, income INT); INSERT INTO farmers_india (id, name, district_id, age, income) VALUES (1, 'Farmer A', 1, 45, 50000); CREATE TABLE districts_india (id INT, name VARCHAR(255), state VARCHAR(255)); INSERT INTO districts_india (id, name, state) VALUES (1, 'District A', 'Maharashtra');
CREATE TABLE Armed_Forces (base_id INT, base_name VARCHAR(50), base_location VARCHAR(50), base_type VARCHAR(50)); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (1, 'Fort Bragg', 'North Carolina', 'Army'); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (2, 'Camp Pendleton', 'California', 'Marines');
```

## 1.olist_orders_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_orders
TABLE_NAME = "orders"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1367, 'output_tokens': 3787, 'total_tokens': 5154, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3008}}
진행중 : 2
{'input_tokens': 1603, 'output_tokens': 4413, 'total_tokens': 6016, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3584}}
진행중 : 3
{'input_tokens': 1876, 'output_tokens': 4611, 'total_tokens': 6487, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3840}}
진행중 : 4
{'input_tokens': 2050, 'output_tokens': 4590, 'total_tokens': 6640, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3712}}
진행중 : 5
{'input_tokens': 2394, 'output_tokens': 4439, 'total_tokens': 6833, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3456}}
진행중 : 6
{'input_tokens': 2624, 'output_tokens': 4127, 'total_toke

In [ ]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
DF_NAME.to_sql(TABLE_NAME, conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

1. Question:   2018년 6월에 구매된 주문 건수
2. SQL 원본:
SELECT COUNT(*) AS order_count
FROM orders
WHERE order_purchase_timestamp >= '2018-06-01' AND order_purchase_timestamp < '2018-07-01'
3. Status:     success

4. 실행 결과:

   order_count
0         6167


--------------------------------------------------
1. Question:   주문 상태별 주문 건수를 알려주나?
2. SQL 원본:
SELECT order_status, COUNT(*) AS cnt
FROM orders
GROUP BY order_status
ORDER BY cnt DESC
3. Status:     success

4. 실행 결과:

  order_status    cnt
0    delivered  96478
1      shipped   1107
2     canceled    625
3  unavailable    609
4     invoiced    314
5   processing    301
6      created      5
7     approved      2


--------------------------------------------------
1. Question:   2017년 12월 25일에 고객에게 배송 완료된 주문의 주문 ID와 고객 ID를 보여주는 것임?
2. SQL 원본:
SELECT order_id, customer_id
FROM orders
WHERE DATE(order_delivered_customer_date) = '2017-12-25'
3. Status:     success

4. 실행 결과:

Empty DataFrame
Columns: [order_id, customer_id]
Index: []


-------

KeyboardInterrupt: 

In [ ]:
#질문 출력해보기
for result in total_result:
  print(result.get('Question'))

2018년 6월에 구매된 주문 건수
주문 상태별 주문 건수를 알려주나?
2017년 12월 25일에 고객에게 배송 완료된 주문의 주문 ID와 고객 ID를 보여주는 것임?
예상 배송일을 넘겨서 도착한 주문 건수
2017년 5월에 구매된 주문들의 결제 승인까지 평균 소요 시간(시간 단위)은 얼마인가요?
2018년 1분기에 구매된 주문 중 배송사에 인계된 주문의 비율은 얼마인가요?
각 고객별 첫 구매 일시는 언제인가요?
2017년에 고객에게 배송 완료된 주문들의 평균 배송 소요일수.
2018년 2월에 고객에게 배송 완료된 주문 중 예상 배송일보다 빨리 도착한 주문의 비율은 얼마인가요?
2017년 9월 10일에 배송사로 인계된 주문의 주문 ID와 인계 시각을 보여주냐?
2017년 상반기에 가장 많이 주문한 상위 5명의 고객 ID와 주문 건수를 알려주실 수 있나요?
2018년 3월에 구매된 주문들의 결제 승인부터 배송사 인계까지 평균 소요 시간(시간 단위)은 얼마입니까?
2018-07-15에 결제 승인됐지만 고객에게 아직 배송 완료되지 않은 주문의 주문 ID를 보여주겠니?
2018년 2분기(4~6월)에 구매된 주문의 상태별 건수를 알려주니?
2017년에 고객에게 배송 완료된 주문 중 예상 배송일을 초과한 주문의 평균 지연 일수
결제 승인 시각보다 먼저 배송사에 인계된 이상 주문은 몇 건인가요?
2017년 월별 구매 건수
고객 수령까지 가장 오래 걸린 주문 상위 10건의 주문 ID와 소요일수(일)을 알려주시나요?
주문 상태가 'unavailable'인 주문들의 주문 ID와 구매 일시를 최신순으로 보여주나요?
2017년 11월 11일에 구매된 주문은 총 몇 건인가요?
2018년 7월에 결제 승인이 난 주문 건수
배송사에 인계됐지만 아직 고객에게 배송 완료되지 않은 주문 건수는 얼마인가요?
2017년 8월 1일부터 10일까지 구매된 주문 중 고객에게 배송 완료된 주문의 비율
예상 배송일보다 빨리 도착한 주문들의 평균 조기 도착 일수는 얼마인가?
2017년 4분기(10~12월)

##2.olist_order_items_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_items
TABLE_NAME = "order_items"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value ; item freight value item (if an order has more than one item the freight value is splitted between items)
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1261, 'output_tokens': 4072, 'total_tokens': 5333, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3136}}
진행중 : 2
{'input_tokens': 1565, 'output_tokens': 3564, 'total_tokens': 5129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 3
{'input_tokens': 1854, 'output_tokens': 4668, 'total_tokens': 6522, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3776}}
진행중 : 4
{'input_tokens': 2156, 'output_tokens': 4731, 'total_tokens': 6887, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 5
{'input_tokens': 2419, 'output_tokens': 4967, 'total_tokens': 7386, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3968}}
진행중 : 6
{'input_tokens': 2755, 'output_tokens': 5212, 'total_toke

##3.olist_order_reviews_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_reviews
TABLE_NAME = "order_reviews"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_reviews (review_id VARCHAR(32) NOT NULL, order_id VARCHAR(32) NOT NULL, review_score INT NOT NULL, review_comment_title VARCHAR(100) NULL, review_comment_message TEXT NULL, review_creation_date DATETIME NOT NULL, review_answer_timestamp DATETIME NOT NULL, PRIMARY KEY (review_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
review_id : unique review identifier
order_id : unique order identifier
review_score : Note ranging from 1 to 5 given by the customer on a satisfaction survey.
review_comment_title : Comment title from the review left by the customer, in Portuguese.
review_comment_message : Comment message from the review left by the customer, in Portuguese.
review_creation_date : Shows the date in which the satisfaction survey was sent to the customer.
review_answer_timestamp : Shows satisfaction survey answer timestamp.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1303, 'output_tokens': 4510, 'total_tokens': 5813, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3648}}
진행중 : 2
{'input_tokens': 1551, 'output_tokens': 6032, 'total_tokens': 7583, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4992}}
진행중 : 3
{'input_tokens': 1958, 'output_tokens': 6378, 'total_tokens': 8336, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5312}}
진행중 : 4
{'input_tokens': 2146, 'output_tokens': 5902, 'total_tokens': 8048, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 5
{'input_tokens': 2447, 'output_tokens': 4824, 'total_tokens': 7271, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4032}}
진행중 : 6
{'input_tokens': 2656, 'output_tokens': 6432, 'total_toke

##4. olist_order_payments_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_order_payments
TABLE_NAME = "order_payments"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE order_payments (order_id VARCHAR(32) NOT NULL, payment_sequential INT NOT NULL, payment_type VARCHAR(20) NOT NULL, payment_installments INT NOT NULL, payment_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, payment_sequential));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of an order.
payment_sequential : a customer may pay an order with more than one payment method. If he does so, a sequence will be created to accommodate all payments.
payment_type : method of payment chosen by the customer.
payment_installments : number of installments chosen by the customer.
payment_value : transaction value.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1099, 'output_tokens': 3381, 'total_tokens': 4480, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 2
{'input_tokens': 1383, 'output_tokens': 5347, 'total_tokens': 6730, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4352}}
진행중 : 3
{'input_tokens': 1652, 'output_tokens': 5841, 'total_tokens': 7493, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 4
{'input_tokens': 1960, 'output_tokens': 5771, 'total_tokens': 7731, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4864}}
진행중 : 5
{'input_tokens': 2290, 'output_tokens': 5748, 'total_tokens': 8038, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4672}}
진행중 : 6
{'input_tokens': 2609, 'output_tokens': 5732, 'total_toke

##5.olist_products_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_products
TABLE_NAME = "products"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE products (product_id VARCHAR(32) NOT NULL, product_category_name VARCHAR(50) NULL, product_name_lenght INT NULL, product_description_lenght INT NULL, product_photos_qty INT NULL, product_weight_g INT NULL, product_length_cm INT NULL, product_height_cm INT NULL, product_width_cm INT NULL, PRIMARY KEY (product_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
product_id : unique product identifier
product_category_name : root category of product, in Portuguese.
product_name_lenght : number of characters extracted from the product name.
product_description_lenght : number of characters extracted from the product description.
product_photos_qty : number of product published photos
product_weight_g : product weight measured in grams.
product_length_cm : product length measured in centimeters.
product_height_cm : product height measured in centimeters.
product_width_cm : product width measured in centimeters.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1193, 'output_tokens': 3619, 'total_tokens': 4812, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2816}}
진행중 : 2
{'input_tokens': 1444, 'output_tokens': 4119, 'total_tokens': 5563, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3072}}
진행중 : 3
{'input_tokens': 1724, 'output_tokens': 5750, 'total_tokens': 7474, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4736}}
진행중 : 4
{'input_tokens': 1936, 'output_tokens': 5082, 'total_tokens': 7018, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4032}}
진행중 : 5
{'input_tokens': 2328, 'output_tokens': 5655, 'total_tokens': 7983, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 6
{'input_tokens': 2623, 'output_tokens': 4347, 'total_toke

##6.olist_sellers_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_sellers
TABLE_NAME = "sellers"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE sellers (seller_id VARCHAR(32) NOT NULL, seller_zip_code_prefix VARCHAR(5) NOT NULL, seller_city VARCHAR(40) NOT NULL, seller_state VARCHAR(2) NOT NULL, PRIMARY KEY (seller_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
seller_id : seller unique identifier
seller_zip_code_prefix : first 5 digits of seller zip code
seller_city : seller city name
seller_state : seller state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")


진행중 : 1
{'input_tokens': 1022, 'output_tokens': 3047, 'total_tokens': 4069, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2432}}
진행중 : 2
{'input_tokens': 1207, 'output_tokens': 3907, 'total_tokens': 5114, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3200}}
진행중 : 3
{'input_tokens': 1468, 'output_tokens': 2849, 'total_tokens': 4317, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2240}}
진행중 : 4
{'input_tokens': 1670, 'output_tokens': 3824, 'total_tokens': 5494, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3200}}
진행중 : 5
{'input_tokens': 1955, 'output_tokens': 5865, 'total_tokens': 7820, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5056}}
진행중 : 6
{'input_tokens': 2169, 'output_tokens': 5783, 'total_toke

##7.olist_order_customer_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_customers
TABLE_NAME = "customers"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE customers (customer_id VARCHAR(32) NOT NULL, customer_unique_id VARCHAR(32) NOT NULL, customer_zip_code_prefix VARCHAR(5) NOT NULL, customer_city VARCHAR(40) NOT NULL, customer_state VARCHAR(2) NOT NULL, PRIMARY KEY (customer_id));
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
customer_id : key to the orders dataset. Each order has a unique customer_id.
customer_unique_id : unique identifier of a customer.
customer_zip_code_prefix : first five digits of customer zip code
customer_city : customer city name
customer_state : customer state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1083, 'output_tokens': 3586, 'total_tokens': 4669, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3008}}
진행중 : 2
{'input_tokens': 1258, 'output_tokens': 3411, 'total_tokens': 4669, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2688}}
진행중 : 3
{'input_tokens': 1536, 'output_tokens': 4358, 'total_tokens': 5894, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3584}}
진행중 : 4
{'input_tokens': 1744, 'output_tokens': 4928, 'total_tokens': 6672, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4160}}
진행중 : 5
{'input_tokens': 2019, 'output_tokens': 6898, 'total_tokens': 8917, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5952}}
진행중 : 6
{'input_tokens': 2208, 'output_tokens': 5223, 'total_toke

##8.olist_geolocation_dataset

In [ ]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME = df_geolocation
TABLE_NAME = "geolocation"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE geolocation (geolocation_zip_code_prefix VARCHAR(5) NOT NULL, geolocation_lat DOUBLE NOT NULL, geolocation_lng DOUBLE NOT NULL, geolocation_city VARCHAR(40) NOT NULL, geolocation_state VARCHAR(2) NOT NULL);
"""

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
geolocation_zip_code_prefix : first 5 digits of zip code
geolocation_lat : latitude
geolocation_lng : longitude
geolocation_city : city name
geolocation_state : state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values = get_sample_values(DF_NAME, 3)

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result
                        )

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement, DF_NAME, TABLE_NAME)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1017, 'output_tokens': 3398, 'total_tokens': 4415, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2560}}
진행중 : 2
{'input_tokens': 1237, 'output_tokens': 4715, 'total_tokens': 5952, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 3
{'input_tokens': 1469, 'output_tokens': 4878, 'total_tokens': 6347, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 4
{'input_tokens': 1717, 'output_tokens': 4359, 'total_tokens': 6076, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3392}}
진행중 : 5
{'input_tokens': 2016, 'output_tokens': 4822, 'total_tokens': 6838, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 6
{'input_tokens': 2202, 'output_tokens': 7040, 'total_toke

# LLM 관련 설정 - 복합

In [22]:
SYSTEM_PROMPT_MULTI = """
#역할
당신은 Text-to-SQL을 수행해야합니다.
DDL 선언문 2개, 칼럼 설명 2개, 칼럼 값 예시 2개, 질문 예시를 참고해 사용자가 할 법한 질문-SQL 쌍을 작성해주세요.
실제 사용자가 Text-to-SQL LLM에게 자연스럽게 물어볼 법한 질문과 그에 정확히 대응하는 SQL을 작성하세요.
제공되는 테이블은 2개입니다. 2개의 테이블을 전부 다 사용하는 예시를 생성하세요.
질문-SQL쌍은 10개 생성하세요.

#최우선 중요 원칙
사람이 실제로 어떻게 질문할지 생각하세요. 그리고 그 질문에 정확히 대응하는 SQL 문을 작성하세요.
질문 예시를 적극적으로 참고하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.
4. SQL 작성시 주의
- BETWEEN : 기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다. 기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'
- 현재 날짜/시간 : 현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.
5. [핵심] 두 테이블을 '진짜로' 사용하는 질문만 생성하세요.
- "두 테이블을 사용한다"는 의미는 SELECT, WHERE, GROUP BY, 계산식 등 어디에서든
  두 테이블의 컬럼이 각각 최소 1개 이상 실제로 쓰여야 한다는 뜻입니다.
- JOIN을 걸었더라도 JOIN한 테이블의 컬럼이 SELECT/WHERE/GROUP BY 어디에도 등장하지 않는다면
  그 JOIN은 불필요한 JOIN입니다. 이런 쿼리는 작성하지 마세요.
- 자가 검증: SQL을 작성한 후 "이 질문이 테이블 하나만으로도 답할 수 있는가?"를 스스로 확인하세요.
  만약 한 테이블만으로 답할 수 있다면, 두 테이블이 모두 필요한 질문으로 바꾸세요.

- 나쁜 예 (orders 컬럼만 쓰고 order_items는 JOIN만 해둔 경우):
  SELECT AVG(DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp))
  FROM orders o
  JOIN order_items i ON i.order_id = o.order_id  -- i.* 컬럼이 어디에도 안 쓰임
  WHERE o.order_status = 'delivered'

- 좋은 예 (두 테이블 컬럼을 모두 실제로 사용):
  SELECT o.order_status, COUNT(*) AS item_count, SUM(i.price) AS total_price
  FROM orders o
  JOIN order_items i ON i.order_id = o.order_id  -- i.price가 SELECT에서 쓰임
  WHERE o.order_purchase_timestamp >= '2018-01-01'
  AND o.order_purchase_timestamp < '2019-01-01'
  GROUP BY o.order_status

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [23]:
def df_to_insert_sql_multi(df1: pd.DataFrame, table_name1: str, df2: pd.DataFrame, table_name2: str) -> tuple[str, str]:
    """
    두 DataFrame에 대해 같은 행 개수(num_rows)를 공유하는 INSERT INTO 문 두 개를 생성한다.
    df_to_insert_sql과 같은 포맷팅 로직을 쓰되, num_rows를 한 번만 뽑아서 두 테이블에 동일하게 적용한다.
    (df_to_insert_sql 자체는 건드리지 않음)

    Args:
        df1, table_name1 : 첫 번째 테이블의 DataFrame / 테이블명
        df2, table_name2 : 두 번째 테이블의 DataFrame / 테이블명

    Returns:
        (table1용 INSERT문, table2용 INSERT문) 튜플. 둘 다 같은 행 개수를 가짐.
    """
    if df1.empty or df2.empty:
        raise ValueError("df가 비어 있어 INSERT 구문을 만들 수 없습니다.")

    # 0~5 사이, 두 df가 다 만들어낼 수 있는 범위 내에서 행 개수를 한 번만 결정
    num_rows = random.randint(0, min(5, len(df1), len(df2)))

    def _build(df, table_name, n):
        if n == 0:
            return ''
        columns = list(df.columns)
        sample_df = df.sample(n=n).reset_index(drop=True)
        is_numeric_col = {col: pd.api.types.is_numeric_dtype(df[col]) for col in columns}

        def format_value(col, val):
            if pd.isna(val):
                return "NULL"
            if is_numeric_col[col] and not isinstance(val, bool):
                return str(val)
            if isinstance(val, bool):
                return str(val).upper()
            escaped = str(val).replace("'", "''")
            return f"'{escaped}'"

        value_rows = []
        for _, row in sample_df.iterrows():
            values = ", ".join(format_value(col, row[col]) for col in columns)
            value_rows.append(f"({values})")

        columns_str = ", ".join(columns)
        values_str = ", ".join(value_rows)
        return f"INSERT INTO {table_name} ({columns_str}) VALUES {values_str};"

    insert_sql1 = _build(df1, table_name1, num_rows)
    insert_sql2 = _build(df2, table_name2, num_rows)

    return insert_sql1, insert_sql2


def convert_to_gretel_format_multi(total_result: list[dict], ddl1: str, df1: pd.DataFrame, table_name1: str,
                                    ddl2: str, df2: pd.DataFrame, table_name2: str) -> list[dict]:
    """
    2개 테이블(DB)을 사용하는 total_result를 gretelai text-to-sql 학습 데이터 형식으로 변환.
    df_to_insert_sql_multi를 사용해 두 테이블의 INSERT 행 개수를 항상 동일하게 맞춘다.

    Args:
        total_result : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
        ddl1         : 첫 번째 테이블의 CREATE TABLE 문
        df1          : 첫 번째 테이블의 INSERT INTO 문을 생성할 DataFrame
        table_name1  : 첫 번째 테이블 명
        ddl2         : 두 번째 테이블의 CREATE TABLE 문
        df2          : 두 번째 테이블의 INSERT INTO 문을 생성할 DataFrame
        table_name2  : 두 번째 테이블 명

    Returns:
        [{'instruction': ..., 'input': '', 'output': ...}, ...]
    """
    converted = []
    for item in total_result:
        question = item['Question']
        sql      = item['SQL']

        insert_sql1, insert_sql2 = df_to_insert_sql_multi(df1, table_name1, df2, table_name2)

        ddl_block = (
            f"{ddl1}\n{insert_sql1}\n"
            f"{ddl2}\n{insert_sql2}"
        )

        instruction = (
            f"입력 텍스트: {question}\n\n"
            f"DDL statements:\n{ddl_block}\n\n"
            f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
        )

        converted.append({
            "instruction": instruction,
            "input"      : "",
            "output"     : f"쿼리 작성: {sql}"
        })

    return converted

#2개 이상 DB를 사용하는 SQL문 생성

## 모음 모음

DDL문 모음
```
CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));

CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (seller_id) REFERENCES sellers(seller_id));

CREATE TABLE order_reviews (review_id VARCHAR(32) NOT NULL, order_id VARCHAR(32) NOT NULL, review_score INT NOT NULL, review_comment_title VARCHAR(100) NULL, review_comment_message TEXT NULL, review_creation_date DATETIME NOT NULL, review_answer_timestamp DATETIME NOT NULL, PRIMARY KEY (review_id), FOREIGN KEY (order_id) REFERENCES orders(order_id));

CREATE TABLE order_payments (order_id VARCHAR(32) NOT NULL, payment_sequential INT NOT NULL, payment_type VARCHAR(20) NOT NULL, payment_installments INT NOT NULL, payment_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, payment_sequential), FOREIGN KEY (order_id) REFERENCES orders(order_id));

CREATE TABLE products (product_id VARCHAR(32) NOT NULL, product_category_name VARCHAR(50) NULL, product_name_lenght INT NULL, product_description_lenght INT NULL, product_photos_qty INT NULL, product_weight_g INT NULL, product_length_cm INT NULL, product_height_cm INT NULL, product_width_cm INT NULL, PRIMARY KEY (product_id));

CREATE TABLE sellers (seller_id VARCHAR(32) NOT NULL, seller_zip_code_prefix VARCHAR(5) NOT NULL, seller_city VARCHAR(40) NOT NULL, seller_state VARCHAR(2) NOT NULL, PRIMARY KEY (seller_id));

CREATE TABLE customers (customer_id VARCHAR(32) NOT NULL, customer_unique_id VARCHAR(32) NOT NULL, customer_zip_code_prefix VARCHAR(5) NOT NULL, customer_city VARCHAR(40) NOT NULL, customer_state VARCHAR(2) NOT NULL, PRIMARY KEY (customer_id));

CREATE TABLE geolocation (geolocation_zip_code_prefix VARCHAR(5) NOT NULL, geolocation_lat DOUBLE NOT NULL, geolocation_lng DOUBLE NOT NULL, geolocation_city VARCHAR(40) NOT NULL, geolocation_state VARCHAR(2) NOT NULL);```

```
#column_descriptions 전체
orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value ; item freight value item (if an order has more than one item the freight value is splitted between items)

order_reviews
review_id : unique review identifier
order_id : unique order identifier
review_score : Note ranging from 1 to 5 given by the customer on a satisfaction survey.
review_comment_title : Comment title from the review left by the customer, in Portuguese.
review_comment_message : Comment message from the review left by the customer, in Portuguese.
review_creation_date : Shows the date in which the satisfaction survey was sent to the customer.
review_answer_timestamp : Shows satisfaction survey answer timestamp.

order_payments
order_id : unique identifier of an order.
payment_sequential : a customer may pay an order with more than one payment method. If he does so, a sequence will be created to accommodate all payments.
payment_type : method of payment chosen by the customer.
payment_installments : number of installments chosen by the customer.
payment_value : transaction value.

products
product_id : unique product identifier
product_category_name : root category of product, in Portuguese.
product_name_lenght : number of characters extracted from the product name.
product_description_lenght : number of characters extracted from the product description.
product_photos_qty : number of product published photos
product_weight_g : product weight measured in grams.
product_length_cm : product length measured in centimeters.
product_height_cm : product height measured in centimeters.
product_width_cm : product width measured in centimeters.

sellers
seller_id : seller unique identifier
seller_zip_code_prefix : first 5 digits of seller zip code
seller_city : seller city name
seller_state : seller state

customers
customer_id : key to the orders dataset. Each order has a unique customer_id.
customer_unique_id : unique identifier of a customer.
customer_zip_code_prefix : first five digits of customer zip code
customer_city : customer city name
customer_state : customer state

geolocation
geolocation_zip_code_prefix : first 5 digits of zip code
geolocation_lat : latitude
geolocation_lng : longitude
geolocation_city : city name
geolocation_state : state
```

## 1. orders & order_items (완)

In [26]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
DF_NAME1 = df_orders
DF_NAME2 = df_order_items
TABLE_NAME1 = "orders"
TABLE_NAME2 = "order_items"

  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement1 = """CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));"""
DDL_Statement2 = """CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (seller_id) REFERENCES sellers(seller_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
두 테이블은 order_id 컬럼으로 연결됩니다(JOIN 가능).

orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value : item freight value item (if an order has more than one item the freight value is splitted between items)
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm_small, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

#해야하는것
#ㅇ 1. 복합 DB SQL 생성용 Prompt 생성하고 검증하기
#####ㅇ 1-1. "정말로"양쪽 DB를 다 써야지 되는 SQL문을 만들도록 PROMPT 고도화
#ㅇ 2. convert_to_gretel_format으로 결과 저장 시 DDL_Statement1, DDL_Statement2 뒤에 각각 INSERT INTO VALUES 붙이도록 함수 수정.
#ㅇ 3. 공통 key가 무엇인지 입력해야하는지 여부

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 2318, 'output_tokens': 5673, 'total_tokens': 7991, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4224}}
진행중 : 2
{'input_tokens': 2640, 'output_tokens': 6754, 'total_tokens': 9394, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5184}}
진행중 : 3
{'input_tokens': 2941, 'output_tokens': 7992, 'total_tokens': 10933, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 6464}}
진행중 : 4
{'input_tokens': 3392, 'output_tokens': 8737, 'total_tokens': 12129, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 7104}}
진행중 : 5
{'input_tokens': 3741, 'output_tokens': 6433, 'total_tokens': 10174, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 4864}}
진행중 : 6
{'input_tokens': 4099, 'output_tokens': 74

In [37]:
for res in total_result[:10]:
  print(res.get('Question'))

2018년 1분기에 구매된 주문들의 주문 상태별로 상품 매출 합계와 배송비 합계를 알려주겠느냐?
2017년 7월에 결제 승인된 주문 기준으로 판매자별 매출 상위 5명.
2018-03-15에 구매된 모든 주문의 총 아이템 수와 총 결제금액.
2018년 월별로, 예상 배송일을 넘겨서 배송된 주문의 비율(%)과 그 달의 총 상품 매출을 알려주는 것이임?
2017년에 구매된 주문들에 대해, 판매자별 결제 승인부터 물류사 인계까지 평균 소요 일수를 알려주겠느냐?
2018년 2분기에 구매되고 상태가 'delivered'인 각 주문의 아이템 수와 총 결제금액(상품가+배송비)
2017년 8월에 배송 마감일이었던 아이템들의 총 상품 매출을 주문 상태별로 알려주시겠어요?
2018년 상반기 고객에게 배송 완료된 주문들에 대한 판매자별 결제 승인부터 고객 배송 완료까지 평균일.
2017년 12월에 구매된 주문 기준으로, 구매 후 2일 이내에 물류사에 인계된 아이템 비중(%)과 판매자별 정보.
2017년에 고객 인도일이 예정일보다 늦었던 주문의 주문 상태별 평균 지연 일수와 총 상품 매출


In [27]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
df_orders.to_sql(TABLE_NAME1, conn, if_exists='replace', index=False)
df_order_items.to_sql(TABLE_NAME2, conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

1. Question:   2018년 1분기에 구매된 주문들의 주문 상태별로 상품 매출 합계와 배송비 합계를 알려주겠느냐?
2. SQL 원본:
SELECT
  o.order_status,
  SUM(i.price) AS total_price,
  SUM(i.freight_value) AS total_freight
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.order_purchase_timestamp >= '2018-01-01'
  AND o.order_purchase_timestamp < '2018-04-01'
GROUP BY o.order_status
3. Status:     success

4. 실행 결과:

  order_status  total_price  total_freight
0     canceled     13019.73        2254.78
1    delivered   2704438.38      460215.73
2     invoiced     16023.14        1335.30
3   processing      6870.66        1038.60
4      shipped     37070.60        7070.75


--------------------------------------------------
1. Question:   2017년 7월에 결제 승인된 주문 기준으로 판매자별 매출 상위 5명.
2. SQL 원본:
SELECT
  i.seller_id,
  SUM(i.price) AS revenue
FROM orders o
JOIN order_items i ON i.order_id = o.order_id
WHERE o.order_approved_at >= '2017-07-01'
  AND o.order_approved_at < '2017-08-01'
GROUP BY i.seller_id
ORDER BY revenue DE

KeyboardInterrupt: 

In [35]:
with open('/content/data/Olist_orders_and_order_items_text_to_sql_data.json') as f:
  data = json.load(f)
print(data[0].get('instruction'))
print(data[0].get('output'))

입력 텍스트: 2018년 1분기에 구매된 주문들의 주문 상태별로 상품 매출 합계와 배송비 합계를 알려주겠느냐?

DDL statements:
CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));
INSERT INTO orders (order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date) VALUES ('59ff850287c170ca0c5844a659a456a0', 'd932de6fce118de65a177f37e11b4520', 'delivered', '2017-09-09 22:59:17', '2017-09-09 23:10:17', '2017-09-12 17:02:48', '2017-09-20 19:08:11', '2017-09-27 00:00:00'), ('8c344483ffcca752c510429c0923b85b', '98656702c7d062513faeeada65ce7bf7', 'delivered', '2017-11-0

##2. orders & order_payments (완)

In [38]:
DF_NAME1 = df_orders
DF_NAME2 = df_order_payments
TABLE_NAME1 = "orders"
TABLE_NAME2 = "order_payments"

DDL_Statement1 = """CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));"""
DDL_Statement2 = """CREATE TABLE order_payments (order_id VARCHAR(32) NOT NULL, payment_sequential INT NOT NULL, payment_type VARCHAR(20) NOT NULL, payment_installments INT NOT NULL, payment_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, payment_sequential), FOREIGN KEY (order_id) REFERENCES orders(order_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 order_id 컬럼으로 연결됩니다(JOIN 가능).

orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_payments
order_id : unique identifier of an order.
payment_sequential : a customer may pay an order with more than one payment method. If he does so, a sequence will be created to accommodate all payments.
payment_type : method of payment chosen by the customer.
payment_installments : number of installments chosen by the customer.
payment_value : transaction value.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)


진행중 : 1
{'input_tokens': 2099, 'output_tokens': 6487, 'total_tokens': 8586, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5120}}
진행중 : 2
{'input_tokens': 2322, 'output_tokens': 6130, 'total_tokens': 8452, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 4608}}
진행중 : 3
{'input_tokens': 2696, 'output_tokens': 8168, 'total_tokens': 10864, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 6656}}
진행중 : 4
{'input_tokens': 3112, 'output_tokens': 9393, 'total_tokens': 12505, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 7680}}
진행중 : 5
{'input_tokens': 3444, 'output_tokens': 7978, 'total_tokens': 11422, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 6080}}
진행중 : 6
{'input_tokens': 3762, 'output_tokens': 85

In [41]:
#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

✅ 100개 변환 완료!


In [43]:
for res in total_result[:50]:
  print(res.get('Question'))

2018년 1분기에 신용카드로 결제된 주문 수와 총 결제 금액이 얼마인가요?
2018년 5월에 구매된 주문 중 할부 없이 결제된 주문들의 평균 결제 총액은 얼마습니까?
결제 수단별로 결제 승인까지의 평균 소요 시간이 몇 시간임?
2018년 6월에 예상 배송일을 넘겨 배송된 주문 중 결제 총액이 500 이상인 주문 ID와 며칠 늦었는지 알려줄 수 있을까?
주문 상태별 주문 수와 총 결제 금액이 어떻게 되느냐?
2018년 2분기 월별 및 결제 수단별 매출액과 주문 건수는 얼마나 되나?
2017년에 결제 수단을 가장 많이 혼합해 사용한 상위 5개 주문과 사용된 결제 수단 개수
2018년 상반기 결제 승인되지 않은 주문의 결제 수단별 결제 시도 금액 합계
2018년에 결제 방식이 단일 결제와 할부 결제인 주문의 건수와 총 결제금액을 비교해 달라는 요청임?
배송 완료된 주문 중 결제 금액 상위 10건과 각 주문의 실제 배송까지 걸린 일수
2017년 4분기 고객별 총 결제 금액 상위 5명과 각 고객의 해당 기간 배송 완료 주문 수
2018년 3월에 구매된 주문의 결제 수단별 결제 건수와 총 결제 금액이 어떻게 되냐?
2017년에 예정일을 넘겨 도착한 주문에 대해 결제 수단별 평균 지연 일수와 주문 수를 알려줄 수 있나?
2018년 8월까지 구매된 주문 중 상태가 배송 완료인 주문의 결제 수단별 총 결제 금액은 얼마냐?
2017년 주문을 대상으로 주문 상태별로 주문 한 건당 사용된 결제 수단의 평균 개수가 얼마인가요?
구매와 취소가 모두 2018년 4월에 이뤄진 주문 중 결제 수단으로 바우처만 사용된 주문 ID를 모두 알려주실 수 있나?
2017년에 할부 10회 이상을 사용한 주문에 대해, 구매월별 실제 배송까지 평균 소요 일수와 주문 수는 얼마임?
2017년 3분기에 결제 총액이 가장 큰 상위 10개 주문의 주문 ID, 구매 일시, 결제 수단 개수.
2017년 월별 총 결제 금액과 주문 수
2018년 상반기에 1회 결제된 결제 건의 결제수단별 건수는 몇 건이며 평균 결제

## 3. orders & order_reviews (완)

In [45]:
DF_NAME1 = df_orders
DF_NAME2 = df_order_reviews
TABLE_NAME1 = "orders"
TABLE_NAME2 = "order_reviews"

DDL_Statement1 = """CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));"""
DDL_Statement2 = """CREATE TABLE order_reviews (review_id VARCHAR(32) NOT NULL, order_id VARCHAR(32) NOT NULL, review_score INT NOT NULL, review_comment_title VARCHAR(100) NULL, review_comment_message TEXT NULL, review_creation_date DATETIME NOT NULL, review_answer_timestamp DATETIME NOT NULL, PRIMARY KEY (review_id), FOREIGN KEY (order_id) REFERENCES orders(order_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 order_id 컬럼으로 연결됩니다(JOIN 가능).

orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_reviews
review_id : unique review identifier
order_id : unique order identifier
review_score : Note ranging from 1 to 5 given by the customer on a satisfaction survey.
review_comment_title : Comment title from the review left by the customer, in Portuguese.
review_comment_message : Comment message from the review left by the customer, in Portuguese.
review_creation_date : Shows the date in which the satisfaction survey was sent to the customer.
review_answer_timestamp : Shows satisfaction survey answer timestamp.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 2264, 'output_tokens': 4732, 'total_tokens': 6996, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3520}}
진행중 : 2
{'input_tokens': 2571, 'output_tokens': 8483, 'total_tokens': 11054, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 6912}}
진행중 : 3
{'input_tokens': 3014, 'output_tokens': 7313, 'total_tokens': 10327, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5696}}
진행중 : 4
{'input_tokens': 3348, 'output_tokens': 6997, 'total_tokens': 10345, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5696}}
진행중 : 5
{'input_tokens': 3772, 'output_tokens': 7025, 'total_tokens': 10797, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5440}}
진행중 : 6
{'input_tokens': 4152, 'output_tokens': 8

## 4. orders & customers (완)

In [46]:
DF_NAME1 = df_orders
DF_NAME2 = df_customers
TABLE_NAME1 = "orders"
TABLE_NAME2 = "customers"

DDL_Statement1 = """CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id), FOREIGN KEY (customer_id) REFERENCES customers(customer_id));"""
DDL_Statement2 = """CREATE TABLE customers (customer_id VARCHAR(32) NOT NULL, customer_unique_id VARCHAR(32) NOT NULL, customer_zip_code_prefix VARCHAR(5) NOT NULL, customer_city VARCHAR(40) NOT NULL, customer_state VARCHAR(2) NOT NULL, PRIMARY KEY (customer_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 customer_id 컬럼으로 연결됩니다(JOIN 가능).
customer_id는 주문마다 새로 발급되는 ID이고, 같은 사람을 식별하려면 customer_unique_id를 사용해야 합니다.

orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

customers
customer_id : key to the orders dataset. Each order has a unique customer_id.
customer_unique_id : unique identifier of a customer.
customer_zip_code_prefix : first five digits of customer zip code
customer_city : customer city name
customer_state : customer state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 2130, 'output_tokens': 4947, 'total_tokens': 7077, 'input_token_details': {'audio': 0, 'cache_read': 1024}, 'output_token_details': {'audio': 0, 'reasoning': 3712}}
진행중 : 2
{'input_tokens': 2390, 'output_tokens': 6462, 'total_tokens': 8852, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 4864}}
진행중 : 3
{'input_tokens': 2735, 'output_tokens': 7026, 'total_tokens': 9761, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5184}}
진행중 : 4
{'input_tokens': 3146, 'output_tokens': 5912, 'total_tokens': 9058, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 4352}}
진행중 : 5
{'input_tokens': 3449, 'output_tokens': 6981, 'total_tokens': 10430, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5120}}
진행중 : 6
{'input_tokens': 3853, 'output_tokens': 8668

## 5. order_items & products (완)

In [47]:
DF_NAME1 = df_order_items
DF_NAME2 = df_products
TABLE_NAME1 = "order_items"
TABLE_NAME2 = "products"

DDL_Statement1 = """CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (seller_id) REFERENCES sellers(seller_id));"""
DDL_Statement2 = """CREATE TABLE products (product_id VARCHAR(32) NOT NULL, product_category_name VARCHAR(50) NULL, product_name_lenght INT NULL, product_description_lenght INT NULL, product_photos_qty INT NULL, product_weight_g INT NULL, product_length_cm INT NULL, product_height_cm INT NULL, product_width_cm INT NULL, PRIMARY KEY (product_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 product_id 컬럼으로 연결됩니다(JOIN 가능).

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value : item freight value item (if an order has more than one item the freight value is splitted between items)

products
product_id : unique product identifier
product_category_name : root category of product, in Portuguese.
product_name_lenght : number of characters extracted from the product name.
product_description_lenght : number of characters extracted from the product description.
product_photos_qty : number of product published photos
product_weight_g : product weight measured in grams.
product_length_cm : product length measured in centimeters.
product_height_cm : product height measured in centimeters.
product_width_cm : product width measured in centimeters.
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 2080, 'output_tokens': 4739, 'total_tokens': 6819, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3584}}
진행중 : 2
{'input_tokens': 2406, 'output_tokens': 6862, 'total_tokens': 9268, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5184}}
진행중 : 3
{'input_tokens': 2841, 'output_tokens': 7509, 'total_tokens': 10350, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 6016}}
진행중 : 4
{'input_tokens': 3190, 'output_tokens': 6902, 'total_tokens': 10092, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 5120}}
진행중 : 5
{'input_tokens': 3676, 'output_tokens': 6305, 'total_tokens': 9981, 'input_token_details': {'audio': 0, 'cache_read': 1280}, 'output_token_details': {'audio': 0, 'reasoning': 4480}}
진행중 : 6
{'input_tokens': 4170, 'output_tokens': 7781, 

## 6. order_items & sellers (완)

In [48]:
DF_NAME1 = df_order_items
DF_NAME2 = df_sellers
TABLE_NAME1 = "order_items"
TABLE_NAME2 = "sellers"

DDL_Statement1 = """CREATE TABLE order_items (order_id VARCHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id VARCHAR(32) NOT NULL, seller_id VARCHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (seller_id) REFERENCES sellers(seller_id));"""
DDL_Statement2 = """CREATE TABLE sellers (seller_id VARCHAR(32) NOT NULL, seller_zip_code_prefix VARCHAR(5) NOT NULL, seller_city VARCHAR(40) NOT NULL, seller_state VARCHAR(2) NOT NULL, PRIMARY KEY (seller_id));"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 seller_id 컬럼으로 연결됩니다(JOIN 가능).

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value : item freight value item (if an order has more than one item the freight value is splitted between items)

sellers
seller_id : seller unique identifier
seller_zip_code_prefix : first 5 digits of seller zip code
seller_city : seller city name
seller_state : seller state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1908, 'output_tokens': 4557, 'total_tokens': 6465, 'input_token_details': {'audio': 0, 'cache_read': 1024}, 'output_token_details': {'audio': 0, 'reasoning': 3456}}
진행중 : 2
{'input_tokens': 2192, 'output_tokens': 5319, 'total_tokens': 7511, 'input_token_details': {'audio': 0, 'cache_read': 1024}, 'output_token_details': {'audio': 0, 'reasoning': 3904}}
진행중 : 3
{'input_tokens': 2539, 'output_tokens': 4952, 'total_tokens': 7491, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 3456}}
진행중 : 4
{'input_tokens': 2919, 'output_tokens': 6300, 'total_tokens': 9219, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 5
{'input_tokens': 3264, 'output_tokens': 5790, 'total_tokens': 9054, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 4160}}
진행중 : 6
{'input_tokens': 3641, 'output_tokens': 7526,

## 7. customers & geolocation (완)

In [49]:
DF_NAME1 = df_customers
DF_NAME2 = df_geolocation
TABLE_NAME1 = "customers"
TABLE_NAME2 = "geolocation"

DDL_Statement1 = """CREATE TABLE customers (customer_id VARCHAR(32) NOT NULL, customer_unique_id VARCHAR(32) NOT NULL, customer_zip_code_prefix VARCHAR(5) NOT NULL, customer_city VARCHAR(40) NOT NULL, customer_state VARCHAR(2) NOT NULL, PRIMARY KEY (customer_id));"""
DDL_Statement2 = """CREATE TABLE geolocation (geolocation_zip_code_prefix VARCHAR(5) NOT NULL, geolocation_lat DOUBLE NOT NULL, geolocation_lng DOUBLE NOT NULL, geolocation_city VARCHAR(40) NOT NULL, geolocation_state VARCHAR(2) NOT NULL);"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 customers.customer_zip_code_prefix와 geolocation.geolocation_zip_code_prefix가 같은 값으로 JOIN 가능합니다.
단, geolocation 쪽은 우편번호당 여러 좌표 행이 있는 1:N 관계라 JOIN 시 중복 행이 생길 수 있습니다.

customers
customer_id : key to the orders dataset. Each order has a unique customer_id.
customer_unique_id : unique identifier of a customer.
customer_zip_code_prefix : first five digits of customer zip code
customer_city : customer city name
customer_state : customer state

geolocation
geolocation_zip_code_prefix : first 5 digits of zip code
geolocation_lat : latitude
geolocation_lng : longitude
geolocation_city : city name
geolocation_state : state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1752, 'output_tokens': 6333, 'total_tokens': 8085, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5056}}
진행중 : 2
{'input_tokens': 2065, 'output_tokens': 6475, 'total_tokens': 8540, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 4800}}
진행중 : 3
{'input_tokens': 2382, 'output_tokens': 7526, 'total_tokens': 9908, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 5952}}
진행중 : 4
{'input_tokens': 2695, 'output_tokens': 9002, 'total_tokens': 11697, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 7168}}
진행중 : 5
{'input_tokens': 3101, 'output_tokens': 7957, 'total_tokens': 11058, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 6400}}
진행중 : 6
{'input_tokens': 3500, 'output_tokens': 11137,

## 8. sellers & geolocation (완)

In [50]:
DF_NAME1 = df_sellers
DF_NAME2 = df_geolocation
TABLE_NAME1 = "sellers"
TABLE_NAME2 = "geolocation"

DDL_Statement1 = """CREATE TABLE sellers (seller_id VARCHAR(32) NOT NULL, seller_zip_code_prefix VARCHAR(5) NOT NULL, seller_city VARCHAR(40) NOT NULL, seller_state VARCHAR(2) NOT NULL, PRIMARY KEY (seller_id));"""
DDL_Statement2 = """CREATE TABLE geolocation (geolocation_zip_code_prefix VARCHAR(5) NOT NULL, geolocation_lat DOUBLE NOT NULL, geolocation_lng DOUBLE NOT NULL, geolocation_city VARCHAR(40) NOT NULL, geolocation_state VARCHAR(2) NOT NULL);"""
DDL_Statement_sum = DDL_Statement1 + "\n" + DDL_Statement2

column_descriptions = """
두 테이블은 sellers.seller_zip_code_prefix와 geolocation.geolocation_zip_code_prefix가 같은 값으로 JOIN 가능합니다.
단, geolocation 쪽은 우편번호당 여러 좌표 행이 있는 1:N 관계라 JOIN 시 중복 행이 생길 수 있습니다.

sellers
seller_id : seller unique identifier
seller_zip_code_prefix : first 5 digits of seller zip code
seller_city : seller city name
seller_state : seller state

geolocation
geolocation_zip_code_prefix : first 5 digits of zip code
geolocation_lat : latitude
geolocation_lng : longitude
geolocation_city : city name
geolocation_state : state
"""

#LLM 실행
total_result = []
for i in range(10):
  print(f'진행중 : {i+1}')

  #3. 칼럼 별 unique한 값 예시
  column_values1 = get_sample_values(DF_NAME1, 3)
  column_values2 = get_sample_values(DF_NAME2, 3)
  column_values_sum = TABLE_NAME1 + "\n" + column_values1 + "\n" + TABLE_NAME2 + "\n" + column_values2

  #4. base dataset으로부터 질문 예시 n개 추출
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement_sum,
                        column_descriptions = column_descriptions,
                        column_values = column_values_sum,
                        question_examples = question_examples,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장. (n = LLM이 한 번에 생성하는 질문-SQL쌍 개수 (Prompt에 적혀있는데 따로 수정할 필요는 없는거같아서 냅둠.))
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

#후처리
#질문 말투 다양화
total_result = rephrase_questions(total_result, llm, 0.4)

#결과 저장하기.
converted_data = convert_to_gretel_format_multi(total_result, DDL_Statement1, DF_NAME1, TABLE_NAME1,
                                                DDL_Statement2, DF_NAME2, TABLE_NAME2)

os.makedirs('data', exist_ok=True)
with open(f'data/Olist_{TABLE_NAME1}_and_{TABLE_NAME2}_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

진행중 : 1
{'input_tokens': 1677, 'output_tokens': 4932, 'total_tokens': 6609, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3776}}
진행중 : 2
{'input_tokens': 1916, 'output_tokens': 6399, 'total_tokens': 8315, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 5056}}
진행중 : 3
{'input_tokens': 2313, 'output_tokens': 8301, 'total_tokens': 10614, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 6784}}
진행중 : 4
{'input_tokens': 2653, 'output_tokens': 7471, 'total_tokens': 10124, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 6016}}
진행중 : 5
{'input_tokens': 2945, 'output_tokens': 7074, 'total_tokens': 10019, 'input_token_details': {'audio': 0, 'cache_read': 1152}, 'output_token_details': {'audio': 0, 'reasoning': 5696}}
진행중 : 6
{'input_tokens': 3370, 'output_tokens': 80

#3. 데이터 평가/검증



```
# 데이터 형식
1. 모든 list의 원소가 dictionary type인지
2. instruction, input, output 키 존재여부
3. instruction과 output의 값 null 여부

# instruction 값
1. 값이 string 형인지 확인
2. 값에 '입력 텍스트', 'DDL statements' 존재여부
3. '입력 텍스트'가 항상 'DDL statements'보다 앞에 오는지

DDL statements에 INSERT문이 있다면
1. CREATE문에 적힌 테이블명과 INSERT 문에 쓰인 테이블명이 일치하는지
2. CREATE문에 적힌 칼럼명과 INSERT 문에 쓰인 칼럼명이 전체 일치하는지
3. INSERT문에 쓰인 칼럼 개수와 값의 개수가 일치하는지.
4. VALUES 값이 칼럼 데이터형에 맞는 올바른 자료형인지
5. VALUES 값이 칼럼 NULL 허용 여부에 맞는지.
6. 전체 INSERT을 봤을 때 PK의 중복 여부

#input 값
1. 항상 빈 문자열인지 체크

# output 값
1. 값이 string 형인지 확인
2. '쿼리 작성' 존재여부
SQL 확인
1. SQL 실제 실행 되는지 여부
2. SQL이 참조하는 컬럼이 DDL statements에 정의된 칼럼인지 여부

#중복 여부
1. 전체 데이터에서 instruction이 중복되는 것이 있는지 확인
2. 전체 데이터에서 output의 쿼리문이 중복되는 것이 있는지 확인

XXXXXXXXXXXXXXXXXXXX
삭제
전체 데이터에서 instruction의 DDL statement가 중복되는 것이 있는지 확인
XXXXXXXXXXXXXXXXXXXX
```

In [ ]:
!pip install sqlglot -q

In [ ]:
import importlib
import validate_dataset

importlib.reload(validate_dataset)
from validate_dataset import validate_json_file

In [ ]:
#전체 통과
report_orders = validate_json_file("data/Olist_orders_text_to_sql_data.json", llm_sql=llm_sql, _convert_to_sqlite=_convert_to_sqlite)
report_order_items = validate_json_file("data/Olist_order_items_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_order_payments = validate_json_file("data/Olist_order_payments_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_products = validate_json_file("data/Olist_products_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_sellers = validate_json_file("data/Olist_sellers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)
report_customers = validate_json_file("data/Olist_customers_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)


파일: data/Olist_orders_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 전체 통과
[중복 여부 - instruction 중복] 전체 통과
[중복 여부 - output SQL 중복] 전체 통과


In [ ]:
report_order_reviews = validate_json_file("data/Olist_order_reviews_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_order_reviews_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[instruction(INSERT) - PRIMARY KEY 값 중복 여부] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 2건
  - index=75 | 1차 오류=no such function: SUBSTRING_INDEX / LLM 변환 후 오류=near "ORDER": syntax error
  - index=79 | 1차 오류

In [ ]:
report_geolocation = validate_json_file("data/Olist_geolocation_text_to_sql_data.json",llm_sql=llm_sql,_convert_to_sqlite=_convert_to_sqlite)

파일: data/Olist_geolocation_text_to_sql_data.json
총 100건 로드

[데이터 형식 - 원소가 dict 타입인지] 전체 통과
[데이터 형식 - instruction/input/output 키 존재] 전체 통과
[데이터 형식 - instruction 값 not null] 전체 통과
[데이터 형식 - output 값 not null] 전체 통과
[instruction 값 - 값이 string 타입인지] 전체 통과
[instruction 값 - '입력 텍스트' 존재] 전체 통과
[instruction 값 - 'DDL statements' 존재] 전체 통과
[instruction 값 - '입력 텍스트'가 'DDL statements'보다 앞에 위치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 테이블명 일치] 전체 통과
[instruction(INSERT) - CREATE/INSERT 컬럼명 전체 일치] 전체 통과
[instruction(INSERT) - 컬럼 개수와 값 개수 일치] 전체 통과
[instruction(INSERT) - VALUES 값이 컬럼 데이터형에 맞는지] 전체 통과
[instruction(INSERT) - VALUES 값이 NULL 허용 여부에 맞는지] 전체 통과
[input 값 - 항상 빈 문자열인지] 전체 통과
[output 값 - 값이 string 타입인지] 전체 통과
[output 값 - '쿼리 작성' 존재] 전체 통과
[output(SQL) - SQL이 참조하는 컬럼이 DDL에 정의되어 있는지] 전체 통과
[output(SQL) - SQL 실제 실행 가능 여부] 위반 2건
  - index=57 | 1차 오류=no such function: STDDEV_POP / LLM 변환 후 오류=no such function: STDEV
  - index=83 | 1차 오류=no such function: STDDEV_SAMP / LLM 변환 후 오류=no such column: